AI Citation:

The research questions and visualization ideas that this data cleaning is a part of is created by us (team members). 

AI was used to help with Pandas syntax and data-cleaning difficulties when needed. 

# 11/10-15 Clean and load data for Project 2 (Quarto dashboard)

## 1. Load, clean, and reshape each CSV file (from wide form to long form (1 row per country per year))

### 1.1. Load Data

In [42]:
import pandas as pd
import glob
import os

# Paths
paths = {
    "education": "data/raw/education.csv",
    "gender":    "data/raw/gender.csv",
    "health":    "data/raw/health.csv",
    "environment": "data/raw/environment.csv",
    "economy":   "data/raw/economy-and-growth.csv",
}

# dfs = {name: pd.read_csv(path) for name, path in paths.items()}

### 1.2. (REMOVED) Inspect column names

In [43]:
# dfs['economy'].columns

In [44]:
# dfs['education'].columns

In [45]:
# dfs['environment'].columns

In [46]:
# dfs['gender'].columns

## 2. A reusable function to extract only the indicators for the countries of interest
This function below helps us load each World Bank CSV (in wide format; Health, Education, Gender, Economy, Environment), filter to our countries + indicators, melt to long form, and filter to the year window. 

**(Note: Long form is used as a stepping stone for filtering.)**

In [47]:
COUNTRIES = [
    "China", "Japan", "Korea, Rep.", "Mongolia", "Hong Kong SAR, China",
    "Indonesia", "Malaysia", "Philippines", "Singapore", "Thailand",
    "Vietnam", "Cambodia", "Lao PDR", "Myanmar", "Brunei Darussalam"
]

YEAR_COLS = [str(y) for y in range(1960, 2021)]

def extract_long(path, code_to_name, start_year=1980, end_year=2021):
    """
    Load a wide World Bank CSV, filter to our countries + indicators,
    melt to long form, and filter to the year window.

    Parameters
    ----------
    path : str
        Path to the raw wide CSV.
    code_to_name : dict
        Mapping {Indicator Code: short_column_name}.
    """
    df = pd.read_csv(path)

    # 1. Filter countries (huge memory saving)
    df = df[df["Country Name"].isin(COUNTRIES)]

    # 2. Filter to the exact indicator codes we want
    df = df[df["Indicator Code"].isin(code_to_name.keys())]

    # 3. Melt wide → long
    year_cols = [c for c in df.columns if c in YEAR_COLS]
    long = df.melt(
        id_vars=["Country Name", "Country Code", "Indicator Code"],
        value_vars=year_cols,
        var_name="Year",
        value_name="value",
    )

    # 4. Type-cast
    long["Year"] = long["Year"].astype(int)
    long["value"] = pd.to_numeric(long["value"], errors="coerce")

    # 5. Trim to the analysis window
    long = long[(long["Year"] >= start_year) & (long["Year"] <= end_year)]

    # 6. Drop missing observations
    long = long.dropna(subset=["value"])

    # 7. Rename indicator codes to short, dashboard-friendly names
    long["indicator"] = long["Indicator Code"].map(code_to_name)

    return long[["Country Name", "Country Code", "Year", "indicator", "value"]]

### 2.1. Apply the extraction to the health file to get life expectancy, infant mortality, and under5 mortality

Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [48]:
# NOTE - these three health indicators have reliable coverage from 1980-2010. 

# Health
health_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/health-raw.csv",
    code_to_name={
        "SP.DYN.LE00.IN": "life_expectancy",
        "SP.DYN.IMRT.IN": "infant_mortality",
        "SH.DYN.MORT":    "under5_mortality",
    },
)

# view data for an example country
# life_exp = health_long[health_long["indicator"]=="life_expectancy"]
# life_exp[life_exp["Country Name"] == "Vietnam"]
# health_long.head()

### 2.2. Apply extraction to Education file to get primary enrollment %, secondary enrollment, and literacy rate
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [49]:
# Education

# 11/15 NOTE while data cleaning:
#   These columns have varied coverage. 
#   secondary_enrollment is relatively stable for many countries, but may be underreported for a handful of more-developed countries (Japan, Singapore, etc.) 


edu_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/education-raw.csv",
    code_to_name={
        "SE.PRM.ENRR":    "primary_enrollment",
        "SE.SEC.ENRR":    "secondary_enrollment",   # fairly reliable; missing for some more-developed countries
        "SE.ADT.LITR.ZS": "literacy_rate",
        "SE.SEC.PROG.ZS": "prog_to_secondary"
    },
)

# Inspect output
# secondary_enrollment = edu_long[edu_long["indicator"] == "prog_to_secondary"]
# secondary_enrollment[secondary_enrollment["Country Name"] == "Malaysia"]

### 2.3. Apply Extraction to Env (Sanitation) to get sanitation_basic_pct, water_basic_pct, water_safe_pct 
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [50]:
env_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/environment-raw.csv",
    code_to_name={
        "SH.STA.BASS.ZS": "sanitation_basic_pct",               # very complete data after 2000; significant (positive) changes for developing countries
        "SH.STA.SMSS.ZS": "sanitation_safely_managed_pct",      # also very complete 2000-2020
        "SH.H2O.BASW.ZS": "water_basic_pct",                    # also very complete 2000-2020
        "SH.STA.ODFC.ZS": "open_defecation_pct",
        # "SH.H2O.SMDW.ZS": "water_safely_managed_pct",           # many missing values (for developing countries)
    }
)

# inspect output
# env[env["Country Code"] == "VNM"]
# env_long[env_long["Country Code"] == "IDN"][env_long["indicator"] == "open_defecation_pct"]

### 2.4. Apply extraction to Economy to get gdp_per_capita and gdp_per_capita_ppp
Use extract_long to load a wide World Bank CSV, filter to our countries + indicators, melt to long form, and filter to the year window.

In [51]:
# Economy
econ_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/economy-and-growth-raw.csv",
    code_to_name={
        "NY.GDP.PCAP.KD": "gdp_per_capita_const2010usd",
        "NY.GDP.PCAP.PP.KD": "gdp_per_capita_PPP",
    },
)

# Check output
# econ_long[econ_long["Country Code"] == "SGP"][econ_long["indicator"] == "gdp_per_capita_PPP"]

### 2.5. Apply extraction to Gender to get adolescent_fertility, married_by_18, teenage_mothers_pct

In [52]:
# Gender (verify codes via the discovery step first)
gender_long = extract_long(
    "C:/dev/fall-2026/DS-2003-Folder/project_2/data/raw/gender-raw.csv",
    code_to_name={
        "SP.ADO.TFRT":          "adolescent_fertility_per_1000_women",    # NOTE: great coverage for countries in 1980-2010
        # "SP.M18.2024.FE.ZS":    "married_by_18_pct",                # Mostly Missing (drop)
        # "SP.MTR.1519.ZS":      "teenage_mothers_pct",               # Mostly Missing (drop)
    },
)

# inspect output
# gender_long[gender_long["Country Code"] == "CHN"][gender_long["indicator"] == "adolescent_fertility_per_1000_women"]

## 3. Build the output by merging the (filtered) long tables by country (wide form; 1 row per country per year; columns are attributes)

### 3.1. Re-pivot the filtered (long) tables to wide form, preparing to join as columns 
(1 column per indicator, 1 row per country per year)

In [53]:
def long_to_wide(long_df):
    return (
        long_df.pivot_table(
            index=["Country Name", "Country Code", "Year"],
            columns="indicator",
            values="value",
            aggfunc="first",   # there should be no duplicates
        )
        .reset_index()
        .rename_axis(None, axis=1)
    )

health_wide  = long_to_wide(health_long)
edu_wide     = long_to_wide(edu_long)
gender_wide  = long_to_wide(gender_long)
econ_wide    = long_to_wide(econ_long)
env_wide     = long_to_wide(env_long)

from functools import reduce
panel = reduce(
    lambda left, right: pd.merge(left, right,
                                 on=["Country Name", "Country Code", "Year"],
                                 how="outer"),
    [health_wide, edu_wide, gender_wide, env_wide, econ_wide],
)

### 3.2. Scaffold the full (Country, Year), and left-join each topic (column/metric/indicator) to it

In other words: Make sure every (country, year) combination from 1980-2020 is present (as one row), even if any rows end up being completely blank (will be dropped later). Then, add each country's entries by column (topic). 

In [54]:
scaffold = pd.MultiIndex.from_product(
    [COUNTRIES, range(1980, 2021)],
    names=["Country Name", "Year"],
).to_frame(index=False)

# Add Country Code via a lookup you already have (from any topic frame)
code_lookup = panel[["Country Name", "Country Code"]].drop_duplicates()
scaffold = scaffold.merge(code_lookup, on="Country Name", how="left")
scaffold = scaffold[["Country Name", "Country Code", "Year"]]

from functools import reduce
panel = reduce(
    lambda left, right: left.merge(right, on=["Country Name", "Country Code", "Year"], how="left"),
    [scaffold, health_wide, edu_wide, gender_wide, env_wide, econ_wide],
)

### Add Region Column (East / SE Asia)

In [55]:
# 1. map countries to regions
REGION_MAP = {
    # East Asia
    "China":                "East Asia",
    "Japan":                "East Asia",
    "Korea, Rep.":          "East Asia",
    "Mongolia":             "East Asia",
    "Hong Kong SAR, China": "East Asia",
    # Southeast Asia
    "Indonesia":            "Southeast Asia",
    "Malaysia":             "Southeast Asia",
    "Philippines":          "Southeast Asia",
    "Singapore":            "Southeast Asia",
    "Thailand":             "Southeast Asia",
    "Vietnam":              "Southeast Asia",
    "Cambodia":             "Southeast Asia",
    "Lao PDR":              "Southeast Asia",
    "Myanmar":              "Southeast Asia",
    "Brunei Darussalam":    "Southeast Asia",
}

# 2. Create a new 'region' column based on the map's value for 'Country Name' (key)
panel["region"] = panel["Country Name"].map(REGION_MAP)

# 3. Fail loudly if a country is unmapped - prevent silent 'NaN'
assert panel["region"].notna().all(), \
    f"Unmapped: {panel.loc[panel['region'].isna(), 'Country Name'].unique()}"

### Add Income Classification Column (Lower, Lower middle, Upper middle, High)

In [56]:
# World Bank income groups (FY2024 / July 2023 list).

# https://datahelpdesk.worldbank.org/knowledgebase/articles/906519
INCOME_MAP = {
    "China":                "Upper middle",
    "Japan":                "High",
    "Korea, Rep.":          "High",
    "Mongolia":             "Upper middle",
    "Hong Kong SAR, China": "High",
    "Indonesia":            "Upper middle",
    "Malaysia":             "Upper middle",
    "Philippines":          "Upper middle",
    "Singapore":            "High",
    "Thailand":             "Upper middle",
    "Vietnam":              "Upper middle",
    "Cambodia":             "Lower middle",
    "Lao PDR":              "Lower middle",
    "Myanmar":              "Lower middle",
    "Brunei Darussalam":    "High",
}

# add income column
panel["income"] = panel["Country Name"].map(INCOME_MAP)

# CHECK if the income column is 100% non-null
assert panel["income"].notna().all(), \
    f"Unmapped: {panel.loc[panel['income'].isna(), 'Country Name'].unique()}"

# CHECK: non-outputted check for whether all country/region/income are present
panel[["Country Name", "region", "income"]].drop_duplicates().sort_values("Country Name")

,Country Name,region,income
574,Brunei Darussalam,Southeast Asia,High
451,Cambodia,Southeast Asia,Lower middle
0,China,East Asia,Upper middle
164,"Hong Kong SAR, China",East Asia,High
205,Indonesia,Southeast Asia,Upper middle
41,Japan,East Asia,High
82,"Korea, Rep.",East Asia,High
492,Lao PDR,Southeast Asia,Lower middle
246,Malaysia,Southeast Asia,Upper middle
123,Mongolia,East Asia,Upper middle


### Reorder columns (so Country Name, Country Code, Region, Income, Year) are first

In [57]:
# 1. get a list of all cols in panel (cleaned data so-far)
cols = panel.columns.tolist()

# 2. desired first (leftmost) columns + other cols
desired_front_cols = [
    "Country Name", 
    "Country Code", 
    "Year", 
    "region", 
    "income",
]
other_cols = [c for c in cols if (c not in desired_front_cols)]

panel = panel[desired_front_cols + other_cols]

Sanity checks after merge (check columns and rows are present)

In [58]:
# 1. Expected number of rows (≈ 15 countries × 31 years, minus any gaps)
print(panel.shape)

# 2. Confirm only our 15 countries
print(sorted(panel["Country Name"].unique()))

# 3. Confirm year range
print(panel["Year"].min(), panel["Year"].max())

# 4. Coverage per indicator (proportion of country-years with a value)
coverage = panel.drop(columns=["Country Name", "Country Code", "Year"]).notna().mean().sort_values()
print(coverage)

panel[panel["Country Code"] == "CHN"]

(615, 19)
['Brunei Darussalam', 'Cambodia', 'China', 'Hong Kong SAR, China', 'Indonesia', 'Japan', 'Korea, Rep.', 'Lao PDR', 'Malaysia', 'Mongolia', 'Myanmar', 'Philippines', 'Singapore', 'Thailand', 'Vietnam']
1980 2020
literacy_rate                          0.128455
prog_to_secondary                      0.339837
sanitation_safely_managed_pct          0.372358
open_defecation_pct                    0.486179
sanitation_basic_pct                   0.489431
water_basic_pct                        0.508943
secondary_enrollment                   0.622764
gdp_per_capita_PPP                     0.749593
primary_enrollment                     0.772358
infant_mortality                       0.910569
under5_mortality                       0.910569
gdp_per_capita_const2010usd            0.962602
life_expectancy                        0.975610
adolescent_fertility_per_1000_women    0.975610
income                                 1.000000
region                                 1.000000
dtype: floa

,Country Name,Country Code,Year,region,income,infant_mortality,life_expectancy,under5_mortality,literacy_rate,primary_enrollment,prog_to_secondary,secondary_enrollment,adolescent_fertility_per_1000_women,open_defecation_pct,sanitation_basic_pct,sanitation_safely_managed_pct,water_basic_pct,gdp_per_capita_PPP,gdp_per_capita_const2010usd
0,China,CHN,1980,East Asia,Upper middle,47.8,66.844,62.2,NaN,109.719368,NaN,43.235531,17.8548,NaN,NaN,NaN,NaN,NaN,347.186732
1,China,CHN,1981,East Asia,Upper middle,46.0,67.260,59.5,NaN,111.169991,NaN,38.991749,18.0774,NaN,NaN,NaN,NaN,NaN,360.292697
2,China,CHN,1982,East Asia,Upper middle,44.5,67.627,57.3,65.505089,110.754158,NaN,33.148258,18.3000,NaN,NaN,NaN,NaN,NaN,387.038702
3,China,CHN,1983,East Asia,Upper middle,43.4,67.949,55.7,NaN,110.778412,NaN,30.290621,19.9864,NaN,NaN,NaN,NaN,NaN,422.573256
4,China,CHN,1984,East Asia,Upper middle,42.7,68.231,54.6,NaN,111.586197,NaN,29.725439,21.6728,NaN,NaN,NaN,NaN,NaN,480.423618
5,China,CHN,1985,East Asia,Upper middle,42.2,68.473,54.0,NaN,116.695183,NaN,31.015779,23.3592,NaN,NaN,NaN,NaN,NaN,537.577510
6,China,CHN,1986,East Asia,Upper middle,42.1,68.673,53.8,NaN,121.483490,NaN,32.632061,25.0456,NaN,NaN,NaN,NaN,NaN,577.043406
7,China,CHN,1987,East Asia,Upper middle,42.1,68.831,53.8,NaN,126.150711,NaN,34.718990,26.7320,NaN,NaN,NaN,NaN,NaN,634.062010
8,China,CHN,1988,East Asia,Upper middle,42.2,68.954,53.9,NaN,128.099396,NaN,36.242180,24.5392,NaN,NaN,NaN,NaN,NaN,693.956594
9,China,CHN,1989,East Asia,Upper middle,42.2,69.054,54.0,NaN,128.474213,63.73277,36.475689,22.3464,NaN,NaN,NaN,NaN,NaN,712.144219


## 4. Final Export
Export the same data to: 1) a csv, and 2) a parquet file (faster loading into Quarto / Jupyter notebook)

In [59]:
panel.to_csv("C:/dev/fall-2026/DS-2003-Folder/project_2/data/clean/panel_1980_2010.csv", index=False)

# parquet for faster loading
panel.to_parquet("C:/dev/fall-2026/DS-2003-Folder/project_2/data/clean/panel_1980_2010.parquet")